<a href="https://colab.research.google.com/github/Aliii2004/hackathon_credit_scoring/blob/main/ml.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚀 ULTIMATE ML PIPELINE - 97%+ ROC AUC

**Maqsad:** ROC AUC ≥ 0.97 ga erishish

**Metodlar:**
1. Kuchli Feature Engineering
2. Multiple Advanced Models
3. Hyperparameter Tuning
4. Ensemble Methods
5. Class Imbalance Handling

**Input:** output/COMPLETE_CLEANED_customer_data.csv

In [ ]:
!pip install lightgbm
!pip install catboost

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: C:\Users\ifut2\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: C:\Users\ifut2\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [ ]:
# ==========================================
# QADAM 1: KUTUBXONALAR
# ==========================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# ML kutubxonalar
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder, RobustScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix, roc_curve
)

# Advanced models
try:
    import xgboost as xgb
    print("✅ XGBoost yuklandi")
except:
    print("⚠️  XGBoost topilmadi: pip install xgboost")

try:
    import lightgbm as lgb
    print("✅ LightGBM yuklandi")
except:
    print("⚠️  LightGBM topilmadi: pip install lightgbm")

try:
    import catboost as cb
    print("✅ CatBoost yuklandi")
except:
    print("⚠️  CatBoost topilmadi: pip install catboost")

# Visualization
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("\n✅ Barcha kutubxonalar tayyor!")

✅ XGBoost yuklandi
✅ LightGBM yuklandi
✅ CatBoost yuklandi

✅ Barcha kutubxonalar tayyor!


## 📂 QADAM 2: MA'LUMOTLARNI YUKLASH

In [ ]:
# ==========================================
# MA'LUMOTLARNI YUKLASH
# ==========================================
print("📂 Ma'lumotlarni yuklash...\n")

try:
    df = pd.read_csv('output/COMPLETE_CLEANED_customer_data.csv')
    print(f"✅ Dataset yuklandi: {df.shape}")
    print(f"   Qatorlar: {df.shape[0]:,}")
    print(f"   Ustunlar: {df.shape[1]}")
    print(f"\n📋 Ustunlar ro'yxati (birinchi 20):")
    print(list(df.columns[:20]))

except FileNotFoundError:
    print("❌ Fayl topilmadi! Avval data cleaning notebook ni ishlatib dataset yarating.")
    df = pd.DataFrame()

if not df.empty:
    # Target variable tekshirish
    if 'default' in df.columns:
        print(f"\n🎯 Target Variable (default):")
        print(df['default'].value_counts())
        print(f"   Default rate: {df['default'].mean()*100:.2f}%")
    else:
        print("\n⚠️  'default' ustuni topilmadi!")

📂 Ma'lumotlarni yuklash...

✅ Dataset yuklandi: (89999, 55)
   Qatorlar: 89,999
   Ustunlar: 55

📋 Ustunlar ro'yxati (birinchi 20):
['customer_id', 'age', 'annual_income', 'employment_length', 'employment_type', 'education', 'marital_status', 'num_dependents', 'application_id', 'application_hour', 'application_day_of_week', 'account_open_year', 'preferred_contact', 'referral_code', 'account_status_code', 'num_login_sessions', 'num_customer_service_calls', 'has_mobile_app', 'paperless_billing', 'default']

🎯 Target Variable (default):
default
0    85405
1     4594
Name: count, dtype: int64
   Default rate: 5.10%


## 🔧 QADAM 3: KUCHLI FEATURE ENGINEERING

In [ ]:
# ==========================================
# FEATURE ENGINEERING
# ==========================================
print("\n🔧 FEATURE ENGINEERING...\n")

if not df.empty:
    # Nusxa olish
    df_fe = df.copy()

    print("1️⃣ Daromad kategoriyalari...")
    if 'annual_income' in df_fe.columns:
        df_fe['income_category'] = pd.cut(df_fe['annual_income'],
                                          bins=[0, 30000, 50000, 70000, 100000, float('inf')],
                                          labels=['Very Low', 'Low', 'Medium', 'High', 'Very High'])
        print("   ✅ income_category")

    print("\n2️⃣ Kredit skori kategoriyalari...")
    if 'credit_score' in df_fe.columns:
        df_fe['credit_category'] = pd.cut(df_fe['credit_score'],
                                          bins=[0, 580, 670, 740, 800, float('inf')],
                                          labels=['Poor', 'Fair', 'Good', 'Very Good', 'Excellent'])
        print("   ✅ credit_category")

    print("\n3️⃣ Yosh guruhlari...")
    if 'age' in df_fe.columns:
        df_fe['age_group'] = pd.cut(df_fe['age'],
                                    bins=[0, 25, 35, 45, 55, float('inf')],
                                    labels=['18-25', '26-35', '36-45', '46-55', '55+'])
        print("   ✅ age_group")

    print("\n4️⃣ DTI risk kategoriyalari...")
    if 'debt_to_income_ratio' in df_fe.columns:
        df_fe['dti_risk'] = pd.cut(df_fe['debt_to_income_ratio'],
                                   bins=[0, 0.2, 0.36, 0.5, float('inf')],
                                   labels=['Low', 'Medium', 'High', 'Critical'])
        print("   ✅ dti_risk")

    print("\n5️⃣ Credit utilization risk...")
    if 'credit_utilization' in df_fe.columns:
        df_fe['util_risk'] = pd.cut(df_fe['credit_utilization'],
                                    bins=[0, 0.3, 0.5, 0.7, float('inf')],
                                    labels=['Low', 'Medium', 'High', 'Critical'])
        print("   ✅ util_risk")

    print("\n6️⃣ Interaction features...")
    # Income to Debt Ratio
    if 'annual_income' in df_fe.columns and 'total_debt_amount' in df_fe.columns:
        df_fe['income_to_debt_ratio'] = df_fe['annual_income'] / (df_fe['total_debt_amount'] + 1)
        print("   ✅ income_to_debt_ratio")

    # Payment Burden
    if 'total_monthly_debt_payment' in df_fe.columns and 'monthly_income' in df_fe.columns:
        df_fe['payment_burden'] = df_fe['total_monthly_debt_payment'] / (df_fe['monthly_income'] + 1)
        print("   ✅ payment_burden")

    # Available Income Ratio
    if 'monthly_free_cash_flow' in df_fe.columns and 'monthly_income' in df_fe.columns:
        df_fe['available_income_ratio'] = df_fe['monthly_free_cash_flow'] / (df_fe['monthly_income'] + 1)
        print("   ✅ available_income_ratio")

    # Debt per Year Employed
    if 'total_debt_amount' in df_fe.columns and 'employment_length' in df_fe.columns:
        df_fe['debt_per_year_employed'] = df_fe['total_debt_amount'] / (df_fe['employment_length'] + 1)
        print("   ✅ debt_per_year_employed")

    print("\n7️⃣ Loan characteristics...")
    # Loan to Income
    if 'loan_amount' in df_fe.columns and 'annual_income' in df_fe.columns:
        df_fe['loan_to_income'] = df_fe['loan_amount'] / (df_fe['annual_income'] + 1)
        print("   ✅ loan_to_income")

    # Monthly Loan Burden
    if 'monthly_payment' in df_fe.columns and 'monthly_income' in df_fe.columns:
        df_fe['monthly_loan_burden'] = df_fe['monthly_payment'] / (df_fe['monthly_income'] + 1)
        print("   ✅ monthly_loan_burden")

    print("\n8️⃣ Account behavior features...")
    # Login per Year
    if 'num_login_sessions' in df_fe.columns and 'account_open_year' in df_fe.columns:
        current_year = datetime.now().year
        df_fe['login_per_year'] = df_fe['num_login_sessions'] / ((current_year - df_fe['account_open_year']) + 1)
        print("   ✅ login_per_year")

    # Service Calls per Year
    if 'num_customer_service_calls' in df_fe.columns and 'account_open_year' in df_fe.columns:
        df_fe['service_calls_per_year'] = df_fe['num_customer_service_calls'] / ((current_year - df_fe['account_open_year']) + 1)
        print("   ✅ service_calls_per_year")

    print("\n9️⃣ Custom Risk Score...")
    # Weighted combination of key risk factors
    risk_components = []
    if 'debt_to_income_ratio' in df_fe.columns:
        risk_components.append(df_fe['debt_to_income_ratio'] * 0.3)
    if 'credit_utilization' in df_fe.columns:
        risk_components.append(df_fe['credit_utilization'] * 0.3)
    if 'payment_to_income_ratio' in df_fe.columns:
        risk_components.append(df_fe['payment_to_income_ratio'] * 0.2)
    if 'loan_to_value_ratio' in df_fe.columns:
        risk_components.append(df_fe['loan_to_value_ratio'] * 0.2)

    if risk_components:
        df_fe['custom_risk_score'] = sum(risk_components)
        print("   ✅ custom_risk_score")

    print(f"\n✅ FEATURE ENGINEERING YAKUNLANDI")
    print(f"   Yangi dataset: {df_fe.shape}")
    print(f"   Qo'shilgan features: {df_fe.shape[1] - df.shape[1]}")


🔧 FEATURE ENGINEERING...

1️⃣ Daromad kategoriyalari...
   ✅ income_category

2️⃣ Kredit skori kategoriyalari...
   ✅ credit_category

3️⃣ Yosh guruhlari...
   ✅ age_group

4️⃣ DTI risk kategoriyalari...
   ✅ dti_risk

5️⃣ Credit utilization risk...
   ✅ util_risk

6️⃣ Interaction features...
   ✅ income_to_debt_ratio
   ✅ payment_burden
   ✅ available_income_ratio
   ✅ debt_per_year_employed

7️⃣ Loan characteristics...
   ✅ loan_to_income
   ✅ monthly_loan_burden

8️⃣ Account behavior features...
   ✅ login_per_year
   ✅ service_calls_per_year

9️⃣ Custom Risk Score...
   ✅ custom_risk_score

✅ FEATURE ENGINEERING YAKUNLANDI
   Yangi dataset: (89999, 69)
   Qo'shilgan features: 14


## 🎯 QADAM 4: TARGET VA FEATURES AJRATISH

In [ ]:
# ==========================================
# TARGET VA FEATURES
# ==========================================
print("\n🎯 TARGET VA FEATURES AJRATISH...\n")

if not df_fe.empty and 'default' in df_fe.columns:
    # Target
    y = df_fe['default']
    print(f"1️⃣ Target (default):")
    print(f"   Shape: {y.shape}")
    print(f"   Distribution: {y.value_counts().to_dict()}")

    # Keraksiz ustunlarni o'chirish
    exclude_cols = [
        'default', 'customer_id', 'application_id',
        'loan_officer_id', 'referral_code',
        'random_noise_1'  # agar mavjud bo'lsa
    ]

    feature_cols = [col for col in df_fe.columns if col not in exclude_cols]
    print(f"\n2️⃣ Features:")
    print(f"   Jami: {len(feature_cols)}")

    # Kategorik ustunlarni encode qilish
    print(f"\n3️⃣ Kategorik ustunlarni encode qilish...")
    X = df_fe[feature_cols].copy()

    categorical_features = X.select_dtypes(include=['object', 'category']).columns
    print(f"   Kategorik ustunlar: {len(categorical_features)}")

    le_dict = {}
    for col in categorical_features:
        le = LabelEncoder()
        X[col] = le.fit_transform(X[col].astype(str))
        le_dict[col] = le
    print(f"   ✅ Encode qilindi")

    # NaN larni tozalash
    print(f"\n4️⃣ NaN larni tozalash...")
    nan_count = X.isnull().sum().sum()
    if nan_count > 0:
        X = X.fillna(X.median())
        print(f"   ✅ {nan_count} ta NaN to'ldirildi")
    else:
        print(f"   ✅ NaN yo'q")

    print(f"\n✅ FINAL FEATURES: {X.shape}")
    print(f"   Qatorlar: {X.shape[0]:,}")
    print(f"   Features: {X.shape[1]}")


🎯 TARGET VA FEATURES AJRATISH...

1️⃣ Target (default):
   Shape: (89999,)
   Distribution: {0: 85405, 1: 4594}

2️⃣ Features:
   Jami: 64

3️⃣ Kategorik ustunlarni encode qilish...
   Kategorik ustunlar: 14
   ✅ Encode qilindi

4️⃣ NaN larni tozalash...
   ✅ NaN yo'q

✅ FINAL FEATURES: (89999, 64)
   Qatorlar: 89,999
   Features: 64


## 📊 QADAM 5: TRAIN/TEST SPLIT VA SCALING

In [ ]:
# ==========================================
# TRAIN/TEST SPLIT
# ==========================================
print("\n📊 TRAIN/TEST SPLIT...\n")

if 'X' in locals() and 'y' in locals():
    # Stratified split (class imbalance uchun)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    print(f"1️⃣ Train/Test split:")
    print(f"   Train: {X_train.shape} ({len(y_train):,} qator)")
    print(f"   Test: {X_test.shape} ({len(y_test):,} qator)")

    print(f"\n2️⃣ Default rate:")
    print(f"   Train: {y_train.mean()*100:.2f}%")
    print(f"   Test: {y_test.mean()*100:.2f}%")

    # Scaling (RobustScaler - outlier ga chidamli)
    print(f"\n3️⃣ Scaling (RobustScaler)...")
    scaler = RobustScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    print(f"   ✅ Scaled")

    # DataFrame ga qaytarish (feature names saqlash uchun)
    X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns, index=X_train.index)
    X_test_scaled = pd.DataFrame(X_test_scaled, columns=X.columns, index=X_test.index)

    print(f"\n✅ TAYYOR TRAIN/TEST SPLIT")


📊 TRAIN/TEST SPLIT...

1️⃣ Train/Test split:
   Train: (71999, 64) (71,999 qator)
   Test: (18000, 64) (18,000 qator)

2️⃣ Default rate:
   Train: 5.10%
   Test: 5.11%

3️⃣ Scaling (RobustScaler)...
   ✅ Scaled

✅ TAYYOR TRAIN/TEST SPLIT


## 🤖 QADAM 6: ADVANCED MODELLAR

In [ ]:
# ==========================================
# MODEL 1: XGBOOST
# ==========================================
print("\n🤖 MODEL 1: XGBOOST...\n")

if 'xgb' in dir():
    # Class imbalance uchun scale_pos_weight
    scale_pos_weight = sum(y_train == 0) / sum(y_train == 1)

    print(f"⚖️  Class imbalance: scale_pos_weight = {scale_pos_weight:.2f}")

    # Hyperparameter tuning
    xgb_params = {
        'max_depth': [6, 8, 10],
        'learning_rate': [0.01, 0.05, 0.1],
        'n_estimators': [200, 300, 500],
        'min_child_weight': [1, 3, 5],
        'subsample': [0.8, 0.9],
        'colsample_bytree': [0.8, 0.9]
    }

    xgb_model = xgb.XGBClassifier(
        random_state=42,
        eval_metric='logloss',
        scale_pos_weight=scale_pos_weight,
        n_jobs=-1
    )

    print("🔍 GridSearchCV (bu biroz vaqt olishi mumkin...)")
    xgb_grid = GridSearchCV(
        xgb_model, xgb_params,
        cv=3, scoring='roc_auc',
        n_jobs=-1, verbose=1
    )

    xgb_grid.fit(X_train, y_train)

    # Best model
    xgb_best = xgb_grid.best_estimator_

    # Predictions
    y_pred_xgb = xgb_best.predict(X_test)
    y_pred_proba_xgb = xgb_best.predict_proba(X_test)[:, 1]

    # Metrics
    xgb_acc = accuracy_score(y_test, y_pred_xgb)
    xgb_roc = roc_auc_score(y_test, y_pred_proba_xgb)
    xgb_precision = precision_score(y_test, y_pred_xgb)
    xgb_recall = recall_score(y_test, y_pred_xgb)
    xgb_f1 = f1_score(y_test, y_pred_xgb)

    print(f"\n✅ XGBOOST RESULTS:")
    print(f"   Accuracy:  {xgb_acc:.4f} ({xgb_acc*100:.2f}%)")
    print(f"   ROC AUC:   {xgb_roc:.4f} {'🎉' if xgb_roc >= 0.97 else ''}")
    print(f"   Precision: {xgb_precision:.4f}")
    print(f"   Recall:    {xgb_recall:.4f}")
    print(f"   F1 Score:  {xgb_f1:.4f}")
    print(f"\n   Best params: {xgb_grid.best_params_}")
else:
    print("⚠️  XGBoost topilmadi!")


🤖 MODEL 1: XGBOOST...

⚖️  Class imbalance: scale_pos_weight = 18.59
🔍 GridSearchCV (bu biroz vaqt olishi mumkin...)
Fitting 3 folds for each of 324 candidates, totalling 972 fits


In [ ]:
# ==========================================
# MODEL 2: LIGHTGBM
# ==========================================
print("\n🤖 MODEL 2: LIGHTGBM...\n")

if 'lgb' in dir():
    # Hyperparameter tuning
    lgb_params = {
        'max_depth': [6, 8, 10],
        'learning_rate': [0.01, 0.05, 0.1],
        'n_estimators': [200, 300, 500],
        'num_leaves': [31, 50, 70],
        'min_child_samples': [20, 30, 50],
        'subsample': [0.8, 0.9],
        'colsample_bytree': [0.8, 0.9]
    }

    lgb_model = lgb.LGBMClassifier(
        random_state=42,
        scale_pos_weight=scale_pos_weight,
        verbose=-1,
        n_jobs=-1
    )

    print("🔍 GridSearchCV...")
    lgb_grid = GridSearchCV(
        lgb_model, lgb_params,
        cv=3, scoring='roc_auc',
        n_jobs=-1, verbose=1
    )

    lgb_grid.fit(X_train, y_train)

    # Best model
    lgb_best = lgb_grid.best_estimator_

    # Predictions
    y_pred_lgb = lgb_best.predict(X_test)
    y_pred_proba_lgb = lgb_best.predict_proba(X_test)[:, 1]

    # Metrics
    lgb_acc = accuracy_score(y_test, y_pred_lgb)
    lgb_roc = roc_auc_score(y_test, y_pred_proba_lgb)
    lgb_precision = precision_score(y_test, y_pred_lgb)
    lgb_recall = recall_score(y_test, y_pred_lgb)
    lgb_f1 = f1_score(y_test, y_pred_lgb)

    print(f"\n✅ LIGHTGBM RESULTS:")
    print(f"   Accuracy:  {lgb_acc:.4f} ({lgb_acc*100:.2f}%)")
    print(f"   ROC AUC:   {lgb_roc:.4f} {'🎉' if lgb_roc >= 0.97 else ''}")
    print(f"   Precision: {lgb_precision:.4f}")
    print(f"   Recall:    {lgb_recall:.4f}")
    print(f"   F1 Score:  {lgb_f1:.4f}")
else:
    print("⚠️  LightGBM topilmadi!")

In [ ]:
# ==========================================
# MODEL 3: CATBOOST
# ==========================================
print("\n🤖 MODEL 3: CATBOOST...\n")

if 'cb' in dir():
    # Hyperparameter tuning
    cb_params = {
        'depth': [6, 8, 10],
        'learning_rate': [0.01, 0.05, 0.1],
        'iterations': [200, 300, 500],
        'l2_leaf_reg': [1, 3, 5]
    }

    cb_model = cb.CatBoostClassifier(
        random_state=42,
        scale_pos_weight=scale_pos_weight,
        verbose=0,
        thread_count=-1
    )

    print("🔍 GridSearchCV...")
    cb_grid = GridSearchCV(
        cb_model, cb_params,
        cv=3, scoring='roc_auc',
        n_jobs=-1, verbose=1
    )

    cb_grid.fit(X_train, y_train)

    # Best model
    cb_best = cb_grid.best_estimator_

    # Predictions
    y_pred_cb = cb_best.predict(X_test)
    y_pred_proba_cb = cb_best.predict_proba(X_test)[:, 1]

    # Metrics
    cb_acc = accuracy_score(y_test, y_pred_cb)
    cb_roc = roc_auc_score(y_test, y_pred_proba_cb)
    cb_precision = precision_score(y_test, y_pred_cb)
    cb_recall = recall_score(y_test, y_pred_cb)
    cb_f1 = f1_score(y_test, y_pred_cb)

    print(f"\n✅ CATBOOST RESULTS:")
    print(f"   Accuracy:  {cb_acc:.4f} ({cb_acc*100:.2f}%)")
    print(f"   ROC AUC:   {cb_roc:.4f} {'🎉' if cb_roc >= 0.97 else ''}")
    print(f"   Precision: {cb_precision:.4f}")
    print(f"   Recall:    {cb_recall:.4f}")
    print(f"   F1 Score:  {cb_f1:.4f}")
else:
    print("⚠️  CatBoost topilmadi!")

In [ ]:
# ==========================================
# MODEL 4: GRADIENT BOOSTING
# ==========================================
print("\n🤖 MODEL 4: GRADIENT BOOSTING...\n")

gb_params = {
    'max_depth': [5, 7, 9],
    'learning_rate': [0.01, 0.05, 0.1],
    'n_estimators': [200, 300, 500],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'subsample': [0.8, 0.9]
}

gb_model = GradientBoostingClassifier(random_state=42)

print("🔍 GridSearchCV...")
gb_grid = GridSearchCV(
    gb_model, gb_params,
    cv=3, scoring='roc_auc',
    n_jobs=-1, verbose=1
)

gb_grid.fit(X_train, y_train)

# Best model
gb_best = gb_grid.best_estimator_

# Predictions
y_pred_gb = gb_best.predict(X_test)
y_pred_proba_gb = gb_best.predict_proba(X_test)[:, 1]

# Metrics
gb_acc = accuracy_score(y_test, y_pred_gb)
gb_roc = roc_auc_score(y_test, y_pred_proba_gb)
gb_precision = precision_score(y_test, y_pred_gb)
gb_recall = recall_score(y_test, y_pred_gb)
gb_f1 = f1_score(y_test, y_pred_gb)

print(f"\n✅ GRADIENT BOOSTING RESULTS:")
print(f"   Accuracy:  {gb_acc:.4f} ({gb_acc*100:.2f}%)")
print(f"   ROC AUC:   {gb_roc:.4f} {'🎉' if gb_roc >= 0.97 else ''}")
print(f"   Precision: {gb_precision:.4f}")
print(f"   Recall:    {gb_recall:.4f}")
print(f"   F1 Score:  {gb_f1:.4f}")

In [ ]:
# ==========================================
# MODEL 5: RANDOM FOREST
# ==========================================
print("\n🤖 MODEL 5: RANDOM FOREST...\n")

rf_params = {
    'n_estimators': [200, 300, 500],
    'max_depth': [10, 15, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2'],
    'class_weight': ['balanced']
}

rf_model = RandomForestClassifier(random_state=42, n_jobs=-1)

print("🔍 GridSearchCV...")
rf_grid = GridSearchCV(
    rf_model, rf_params,
    cv=3, scoring='roc_auc',
    n_jobs=-1, verbose=1
)

rf_grid.fit(X_train, y_train)

# Best model
rf_best = rf_grid.best_estimator_

# Predictions
y_pred_rf = rf_best.predict(X_test)
y_pred_proba_rf = rf_best.predict_proba(X_test)[:, 1]

# Metrics
rf_acc = accuracy_score(y_test, y_pred_rf)
rf_roc = roc_auc_score(y_test, y_pred_proba_rf)
rf_precision = precision_score(y_test, y_pred_rf)
rf_recall = recall_score(y_test, y_pred_rf)
rf_f1 = f1_score(y_test, y_pred_rf)

print(f"\n✅ RANDOM FOREST RESULTS:")
print(f"   Accuracy:  {rf_acc:.4f} ({rf_acc*100:.2f}%)")
print(f"   ROC AUC:   {rf_roc:.4f} {'🎉' if rf_roc >= 0.97 else ''}")
print(f"   Precision: {rf_precision:.4f}")
print(f"   Recall:    {rf_recall:.4f}")
print(f"   F1 Score:  {rf_f1:.4f}")

## 🏆 QADAM 7: ENSEMBLE METHOD

In [ ]:
# ==========================================
# WEIGHTED ENSEMBLE
# ==========================================
print("\n🏆 WEIGHTED ENSEMBLE...\n")

# Barcha modellar mavjudligini tekshirish
available_predictions = []
weights = []

if 'y_pred_proba_xgb' in locals():
    available_predictions.append(y_pred_proba_xgb)
    weights.append(0.3)
    print("✅ XGBoost qo'shildi (weight: 0.3)")

if 'y_pred_proba_lgb' in locals():
    available_predictions.append(y_pred_proba_lgb)
    weights.append(0.25)
    print("✅ LightGBM qo'shildi (weight: 0.25)")

if 'y_pred_proba_cb' in locals():
    available_predictions.append(y_pred_proba_cb)
    weights.append(0.25)
    print("✅ CatBoost qo'shildi (weight: 0.25)")

if 'y_pred_proba_gb' in locals():
    available_predictions.append(y_pred_proba_gb)
    weights.append(0.15)
    print("✅ Gradient Boosting qo'shildi (weight: 0.15)")

if 'y_pred_proba_rf' in locals():
    available_predictions.append(y_pred_proba_rf)
    weights.append(0.05)
    print("✅ Random Forest qo'shildi (weight: 0.05)")

if available_predictions:
    # Normalize weights
    weights = np.array(weights)
    weights = weights / weights.sum()

    # Weighted average
    ensemble_proba = np.average(available_predictions, axis=0, weights=weights)
    ensemble_pred = (ensemble_proba > 0.5).astype(int)

    # Metrics
    ensemble_acc = accuracy_score(y_test, ensemble_pred)
    ensemble_roc = roc_auc_score(y_test, ensemble_proba)
    ensemble_precision = precision_score(y_test, ensemble_pred)
    ensemble_recall = recall_score(y_test, ensemble_pred)
    ensemble_f1 = f1_score(y_test, ensemble_pred)

    print(f"\n✅ ENSEMBLE RESULTS:")
    print(f"   Accuracy:  {ensemble_acc:.4f} ({ensemble_acc*100:.2f}%)")
    print(f"   ROC AUC:   {ensemble_roc:.4f} {'🎉🎉🎉' if ensemble_roc >= 0.97 else '🎉' if ensemble_roc >= 0.95 else ''}")
    print(f"   Precision: {ensemble_precision:.4f}")
    print(f"   Recall:    {ensemble_recall:.4f}")
    print(f"   F1 Score:  {ensemble_f1:.4f}")
else:
    print("⚠️  Hech qanday model topilmadi!")

## 📊 QADAM 8: NATIJALARNI TAQQOSLASH

In [ ]:
# ==========================================
# NATIJALARNI TO'PLASH
# ==========================================
print("\n📊 BARCHA NATIJALAR:\n" + "="*80)

results = {}

if 'xgb_roc' in locals():
    results['XGBoost'] = {
        'Accuracy': xgb_acc,
        'ROC AUC': xgb_roc,
        'Precision': xgb_precision,
        'Recall': xgb_recall,
        'F1 Score': xgb_f1
    }

if 'lgb_roc' in locals():
    results['LightGBM'] = {
        'Accuracy': lgb_acc,
        'ROC AUC': lgb_roc,
        'Precision': lgb_precision,
        'Recall': lgb_recall,
        'F1 Score': lgb_f1
    }

if 'cb_roc' in locals():
    results['CatBoost'] = {
        'Accuracy': cb_acc,
        'ROC AUC': cb_roc,
        'Precision': cb_precision,
        'Recall': cb_recall,
        'F1 Score': cb_f1
    }

if 'gb_roc' in locals():
    results['Gradient Boosting'] = {
        'Accuracy': gb_acc,
        'ROC AUC': gb_roc,
        'Precision': gb_precision,
        'Recall': gb_recall,
        'F1 Score': gb_f1
    }

if 'rf_roc' in locals():
    results['Random Forest'] = {
        'Accuracy': rf_acc,
        'ROC AUC': rf_roc,
        'Precision': rf_precision,
        'Recall': rf_recall,
        'F1 Score': rf_f1
    }

if 'ensemble_roc' in locals():
    results['Ensemble'] = {
        'Accuracy': ensemble_acc,
        'ROC AUC': ensemble_roc,
        'Precision': ensemble_precision,
        'Recall': ensemble_recall,
        'F1 Score': ensemble_f1
    }

# DataFrame yaratish
results_df = pd.DataFrame(results).T
results_df = results_df.sort_values('ROC AUC', ascending=False)

print(results_df.to_string())

# Eng yaxshi model
best_model = results_df.index[0]
best_roc = results_df.loc[best_model, 'ROC AUC']

print("\n" + "="*80)
print(f"🏆 ENG YAXSHI MODEL: {best_model}")
print(f"   ROC AUC: {best_roc:.4f}")
print(f"   Accuracy: {results_df.loc[best_model, 'Accuracy']:.4f}")

if best_roc >= 0.97:
    print("\n🎉🎉🎉 MAQSADGA YETILDI! ROC AUC ≥ 0.97 🎉🎉🎉")
elif best_roc >= 0.95:
    print(f"\n🎉 Juda yaxshi! ROC AUC = {best_roc:.4f} (maqsad: 0.97)")
    print("   Qo'shimcha optimization kerak bo'lishi mumkin")
else:
    print(f"\n⚠️  ROC AUC = {best_roc:.4f} (maqsad: 0.97)")
    print("   Qo'shimcha feature engineering yoki SMOTE ishlatib ko'ring")

print("="*80)

## 📈 QADAM 9: VIZUALIZATSIYA

In [ ]:
# ==========================================
# VIZUALIZATSIYA
# ==========================================
print("\n📈 VIZUALIZATSIYA...\n")

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('ML MODEL PERFORMANCE COMPARISON', fontsize=16, fontweight='bold')

# 1. ROC AUC Comparison
ax1 = axes[0, 0]
results_df['ROC AUC'].sort_values().plot(kind='barh', ax=ax1, color='skyblue')
ax1.set_title('ROC AUC Comparison')
ax1.set_xlabel('ROC AUC Score')
ax1.axvline(x=0.97, color='red', linestyle='--', label='Target (0.97)')
ax1.legend()

# 2. Accuracy Comparison
ax2 = axes[0, 1]
results_df['Accuracy'].sort_values().plot(kind='barh', ax=ax2, color='lightgreen')
ax2.set_title('Accuracy Comparison')
ax2.set_xlabel('Accuracy Score')

# 3. F1 Score Comparison
ax3 = axes[0, 2]
results_df['F1 Score'].sort_values().plot(kind='barh', ax=ax3, color='lightcoral')
ax3.set_title('F1 Score Comparison')
ax3.set_xlabel('F1 Score')

# 4. Confusion Matrix (Best Model)
ax4 = axes[1, 0]
if best_model == 'Ensemble':
    cm = confusion_matrix(y_test, ensemble_pred)
elif best_model == 'XGBoost':
    cm = confusion_matrix(y_test, y_pred_xgb)
elif best_model == 'CatBoost':
    cm = confusion_matrix(y_test, y_pred_cb)
elif best_model == 'Gradient Boosting':
    cm = confusion_matrix(y_test, y_pred_gb)
else:
    cm = confusion_matrix(y_test, y_pred_rf)

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax4,
            xticklabels=['No Default', 'Default'],
            yticklabels=['No Default', 'Default'])
ax4.set_title(f'Confusion Matrix ({best_model})')
ax4.set_ylabel('Actual')
ax4.set_xlabel('Predicted')

# 5. ROC Curve (Best Model)
ax5 = axes[1, 1]
if best_model == 'Ensemble':
    fpr, tpr, _ = roc_curve(y_test, ensemble_proba)
elif best_model == 'XGBoost':
    fpr, tpr, _ = roc_curve(y_test, y_pred_proba_xgb)
elif best_model == 'LightGBM':
    fpr, tpr, _ = roc_curve(y_test, y_pred_proba_lgb)
elif best_model == 'CatBoost':
    fpr, tpr, _ = roc_curve(y_test, y_pred_proba_cb)
elif best_model == 'Gradient Boosting':
    fpr, tpr, _ = roc_curve(y_test, y_pred_proba_gb)
else:
    fpr, tpr, _ = roc_curve(y_test, y_pred_proba_rf)

ax5.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {best_roc:.4f})')
ax5.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random')
ax5.set_xlim([0.0, 1.0])
ax5.set_ylim([0.0, 1.05])
ax5.set_xlabel('False Positive Rate')
ax5.set_ylabel('True Positive Rate')
ax5.set_title(f'ROC Curve ({best_model})')
ax5.legend(loc="lower right")
ax5.grid(True, alpha=0.3)

# 6. Precision-Recall Comparison
ax6 = axes[1, 2]
metrics_comparison = results_df[['Precision', 'Recall']]
metrics_comparison.plot(kind='bar', ax=ax6, rot=45)
ax6.set_title('Precision vs Recall')
ax6.set_ylabel('Score')
ax6.legend()
ax6.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('output/ml_performance_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Grafiklar saqlandi: output/ml_performance_comparison.png")

## 📋 QADAM 10: FEATURE IMPORTANCE

In [ ]:
# ==========================================
# FEATURE IMPORTANCE (Eng yaxshi model uchun)
# ==========================================
print("\n📋 FEATURE IMPORTANCE...\n")

# Eng yaxshi modelni tanlash
if best_model == 'XGBoost' and 'xgb_best' in locals():
    best_model_obj = xgb_best
elif best_model == 'LightGBM' and 'lgb_best' in locals():
    best_model_obj = lgb_best
elif best_model == 'CatBoost' and 'cb_best' in locals():
    best_model_obj = cb_best
elif best_model == 'Gradient Boosting' and 'gb_best' in locals():
    best_model_obj = gb_best
elif best_model == 'Random Forest' and 'rf_best' in locals():
    best_model_obj = rf_best
else:
    # Ensemble uchun birinchi tree-based modelni ishlatish
    if 'xgb_best' in locals():
        best_model_obj = xgb_best
        print("⚠️  Ensemble uchun XGBoost feature importance ko'rsatiladi")
    elif 'lgb_best' in locals():
        best_model_obj = lgb_best
        print("⚠️  Ensemble uchun LightGBM feature importance ko'rsatiladi")
    else:
        best_model_obj = None

if best_model_obj and hasattr(best_model_obj, 'feature_importances_'):
    # Feature importance dataframe
    feature_imp = pd.DataFrame({
        'feature': X.columns,
        'importance': best_model_obj.feature_importances_
    }).sort_values('importance', ascending=False)

    print(f"📊 TOP 20 MUHIM FEATURES ({best_model}):")
    print(feature_imp.head(20).to_string(index=False))

    # Visualizatsiya
    plt.figure(figsize=(12, 8))
    feature_imp.head(20).plot(x='feature', y='importance', kind='barh', color='teal')
    plt.title(f'Top 20 Feature Importance ({best_model})', fontsize=14, fontweight='bold')
    plt.xlabel('Importance Score')
    plt.ylabel('Features')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.savefig('output/feature_importance.png', dpi=300, bbox_inches='tight')
    plt.show()

    # CSV ga saqlash
    feature_imp.to_csv('output/feature_importance.csv', index=False)
    print("\n✅ Feature importance saqlandi: output/feature_importance.csv")
else:
    print("⚠️  Feature importance mavjud emas")

## 💾 QADAM 11: NATIJALARNI SAQLASH

In [ ]:
# ==========================================
# NATIJALARNI SAQLASH
# ==========================================
print("\n💾 NATIJALARNI SAQLASH...\n")

# 1. Model natijalarini CSV ga saqlash
results_df.to_csv('output/ml_results_final.csv')
print("✅ Model natijalari: output/ml_results_final.csv")

# 2. Detailed report yaratish
timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

report = f"""
{'='*80}
ML MODEL PERFORMANCE REPORT
{'='*80}

Generated: {timestamp}
Dataset: output/COMPLETE_CLEANED_customer_data.csv

{'='*80}
DATASET INFO
{'='*80}
Total Samples: {len(df):,}
Total Features: {X.shape[1]}
Train Samples: {len(y_train):,}
Test Samples: {len(y_test):,}
Default Rate (Train): {y_train.mean()*100:.2f}%
Default Rate (Test): {y_test.mean()*100:.2f}%

{'='*80}
MODEL RESULTS
{'='*80}
"""

for model_name in results_df.index:
    report += f"\n{model_name}:\n"
    report += f"  Accuracy:  {results_df.loc[model_name, 'Accuracy']:.4f}\n"
    report += f"  ROC AUC:   {results_df.loc[model_name, 'ROC AUC']:.4f}\n"
    report += f"  Precision: {results_df.loc[model_name, 'Precision']:.4f}\n"
    report += f"  Recall:    {results_df.loc[model_name, 'Recall']:.4f}\n"
    report += f"  F1 Score:  {results_df.loc[model_name, 'F1 Score']:.4f}\n"

report += f"""
{'='*80}
BEST MODEL
{'='*80}
Model: {best_model}
ROC AUC: {best_roc:.4f}
Accuracy: {results_df.loc[best_model, 'Accuracy']:.4f}

Status: {'✅ MAQSADGA YETILDI (ROC AUC ≥ 0.97)' if best_roc >= 0.97 else '⚠️  QOSHIMCHA OPTIMIZATION KERAK'}

{'='*80}
GENERATED FILES
{'='*80}
1. output/ml_results_final.csv - Model comparison
2. output/ml_performance_comparison.png - Visualizations
3. output/feature_importance.csv - Feature rankings
4. output/feature_importance.png - Feature visualization
5. output/ml_report.txt - This report

{'='*80}
END OF REPORT
{'='*80}
"""

# Report ni saqlash
with open('output/ml_report.txt', 'w', encoding='utf-8') as f:
    f.write(report)

print(report)
print("\n✅ Detailed report: output/ml_report.txt")

# 3. Best model ni pickle ga saqlash (ixtiyoriy)
try:
    import joblib

    if 'best_model_obj' in locals() and best_model_obj:
        joblib.dump(best_model_obj, f'output/best_model_{best_model.replace(" ", "_")}.pkl')
        print(f"\n✅ Best model saqlandi: output/best_model_{best_model.replace(' ', '_')}.pkl")

        # Scaler ham saqlash
        joblib.dump(scaler, 'output/scaler.pkl')
        print("✅ Scaler saqlandi: output/scaler.pkl")
except:
    print("\n⚠️  joblib topilmadi - model saqlanmadi")

## 🎯 QADAM 12: YAKUNIY XULOSALAR

In [ ]:
# ==========================================
# YAKUNIY XULOSALAR
# ==========================================
print("\n" + "="*80)
print("🎯 YAKUNIY XULOSALAR")
print("="*80)

print(f"\n1️⃣ MAQSAD: ROC AUC ≥ 0.97")
print(f"   Natija: {best_roc:.4f}")

if best_roc >= 0.97:
    print("   ✅ MAQSADGA YETILDI! 🎉🎉🎉")
    print("\n   🏆 Tabriklaymiz! Sizning modelingiz juda yaxshi natija ko'rsatdi!")
elif best_roc >= 0.95:
    print(f"   🎉 Juda yaxshi! (maqsadga {0.97 - best_roc:.4f} yetishmayapti)")
    print("\n   💡 Qo'shimcha yaxshilash uchun tavsiyalar:")
    print("      • SMOTE (oversampling) ishlatib ko'ring")
    print("      • Threshold tuning (0.5 dan boshqa)")
    print("      • Stacking ensemble")
    print("      • Qo'shimcha feature engineering")
else:
    print(f"   ⚠️  Qo'shimcha ish kerak (maqsadga {0.97 - best_roc:.4f} yetishmayapti)")
    print("\n   💡 Tavsiyalar:")
    print("      • Data quality tekshiring")
    print("      • Outlier detection")
    print("      • Feature selection")
    print("      • SMOTE yoki ADASYN ishlatish")
    print("      • Cross-validation natijalarini tahlil qiling")

print(f"\n2️⃣ ENG YAXSHI MODEL: {best_model}")
print(f"   • ROC AUC: {best_roc:.4f}")
print(f"   • Accuracy: {results_df.loc[best_model, 'Accuracy']:.4f}")
print(f"   • F1 Score: {results_df.loc[best_model, 'F1 Score']:.4f}")

print(f"\n3️⃣ SAQLANGAN FAYLLAR:")
print(f"   • output/ml_results_final.csv")
print(f"   • output/ml_performance_comparison.png")
print(f"   • output/feature_importance.csv")
print(f"   • output/ml_report.txt")

print("\n" + "="*80)
print("✅ ML PIPELINE YAKUNLANDI!")
print("="*80)

print("\n🚀 Keyingi qadamlar:")
print("   1. Model natijalarini tahlil qiling")
print("   2. Feature importance ni o'rganing")
print("   3. Zarur bo'lsa, qo'shimcha optimization qiling")
print("   4. Production uchun modelni deploy qiling")
print("\n💡 Omad! 🎯")